# LLM-агенты и агентские системы

Языковая модель — это просто функция «текст на входе => текст на выходе». __Агент__ — это та же модель, помещённая в цикл, где она рассуждает, действует через инструменты, наблюдает результат и повторяет шаги, пока не достигнет цели. Грубо говоря, агент - это LLM, получившая некое подобие субъектности 

История развития области — это история того, как мы постепенно достраивали этот цикл: сначала научились заставлять модель думать, потом — связали мышление с действием, потом — начали координировать множество таких шагов и делать всё это надёжно и дёшево в продакшене

<img src="img/agent.jpg" width=500>

## Рассуждения (Reasoning)
Ранние LLM работали по принципу «вопрос => ответ» и если на простых задачах это хорошо работает, то на задачах, требующих многошагового логического рассуждения, модель часто ошибается, пытаясь сразу «перепрыгнуть» к ответу. Оказалось, что дело не столько в нехватке знаний, сколько в отсутствии у модели «черновика»: не было пространства, чтобы развернуть ход мыслию. __Reasoning__ - это способность модели не выдавать ответ сразу, а разворачивать промежуточные шаги рассуждения перед финальным выводом.

В 2022 году авторы из Google [[Wei et al., 2022]](https://arxiv.org/abs/2201.11903) обнаружили, что если попросить модель не выдавать ответ сразу, а сначала проговаривать промежуточные шаги, ее точность на сложных логических задачах резко растёт. Они это делали через Few-shot промптинг: модели явно показывали пример, какого формата они ожидают ответ. Этот вид промптинга назвали **Chain-of-Thought (CoT)** размышление

Промежуточные шаги предоставляют пространство, в котором сложная задача декомпозируется на простые подзадачи. Кроме того, каждый дополнительный токен рассуждения — это, дополнительная порция вычислений, потраченная на задачу

Лучше всего для тестирования рассуждения подходят математические задачи - есть строгое поэтапное ветвление логики и один объективный конечный ответ

Чуть позже ([Kojima et al., 2022](https://arxiv.org/abs/2205.11916)) показали, что даже не нужно добавлять в промпт few shot примеры, достаточно одной фразы вроде «давай рассуждать шаг за шагом», чтобы запустить тот же самый эффект. Это поразительно, но одна эта фраза увеличивает точность модели в 2-4 раза на логических задачах. А с примерами прирост еще выше

Важно отметить, что CoT помогает только там, где есть что декомпозировать (арифметика, логика, многоступенчатые выводы), но может вредить на простых задачах, где лишние рассуждения лишь добавляют шум и возможности ошибиться.

Одна цепочка рассуждений ненадёжна: модель может уверенно пойти по ошибочному пути. ([Wang et al., 2022](https://arxiv.org/abs/2203.11171)) предлагает генерировать не одну цепочку рассуждений, а сразу много, за счёт случайности при сэмплировании. затем выбрать ответ голосованием большинства. Ключевая интуиция: правильное рассуждение обычно сходится к одному и тому же ответу разными путями, тогда как ошибки «расходятся» и не складываются в большинство. Такой подход назвали __Self-Consistency__ размышление

Работа ([Yao et al., 2023](https://arxiv.org/abs/2305.10601)) идёт дальше и переосмысляет рассуждение как поиск по дереву. Вместо линейной цепочки модель на каждом шаге порождает несколько вариантов продолжения рассуждения, оценивает их перспективность, углубляется в удачные ветви и откатывается от тупиковых

<img src="img/cot4.png" width=500>

В промпте модель явно просят сделать ровно один логический шаг, поэтому она понимала, когда ей нужно остановиться для переоценки хода решения. В роли оценщика (Verifier или Model-as-a-judge) обычно выступает та же самая модель, она отвечает на вопрос: похоже ли, что рассуждение приведет к правильному ответу? 

Авторы пробовали два режима измерения: в абсолютах ("sure" - да, уже пришли к ответу, "impossible" - ветка рассуждения плохая ни к чему не приведет, "maybe" - пока нельзя сказать) или отранжировать (отсортируй по потенциалу и мы отберем топ-5 перспективных цепочек). 

Тут правда возникает проблема предвзятости модели (Self-Correction Fallacy) при оценке своего результата. Во-первых, ИИ плохо калибрует свою уверенность (в целом, как и люди). Во-вторых, у ИИ есть склонность "поддакивать" пользователю, если контексте есть утверждение X, модель считает, что логичнее ему следовать, чем опровергать

Как с этой проблемой боролись авторы:<br>- для уменьшения "шума" просили повторить оценку каждой цепочки k раз и затем агрегировали результаты с помощью голосования (majority vote)<br>- грамотно составляли промпты на оценку: не "как ты оцениваешь рассуждение?", а просили проделать аналитическубю работу, провести тесты и сравнения<br>
Что в общем случае можно еще сделать:<br>- разделить "модель генератор" и "модель критика"<br>В соверменных моделях используется Reinforcement Learning

Как делали селекцию вариантов для продолжения
- Beam Search: после каждого этапа генерации оставляют не более K наиболее перспективных вариантов продолжений
- Breadth First Search: на каждом шаге выкидываем только те ветки, которые оцениваются моделью как "impossible"

Кроме того глубину дерева ограничивали, чтобы оно в бесконечном цикле не ветвилось

И в Self-Consistency, и в Tree-of-thought начинает явно выражаться компромисс: качество против стоимости вычислений. Становится понятно, что точность можно буквально «покупать» дополнительными вычислениями на этапе инференса, а не только на этапе обучения модели

### Program-of-Thoughts (PoT)
Вместо плана пишется код

### Auto-CoT
Для генерации few-shot примеров, когда нет желания или возможности собирать примеры вручную

### Test-Time Scaling

За следующую пару лет 2024–2025 подход качественно изменился. Если раньше рассуждение «выманивали» промптом у обычной модели, то теперь модель специально обучают рассуждать - появился целый класс reasoning-моделей

Центральная концепция — масштабирование на инференсе (test-time compute / inference-time scaling). Это новая ось масштабирования, дополняющая привычное масштабирование на предобучении: оказалось, что качество растёт, если позволить модели «думать» дольше — порождать длинные внутренние рассуждения перед ответом. Отсюда и «думающие» режимы, где модель тратит переменный бюджет вычислений в зависимости от сложности задачи.

Как этому обучают — отдельный сюжет. Ключевой приём — RL с проверяемыми наградами (RLVR, RL with verifiable rewards): модель тренируют на задачах, где правильность ответа можно автоматически проверить (математика, код), и награждают за верный итоговый результат, не предписывая, *как именно* рассуждать. Модель сама «открывает» эффективные стратегии рассуждения 

### DeepSeek Reasoning
Яркий пример подхода к рассуждениям - модель [DeepSeek-R1, 2025](https://arxiv.org/abs/2501.12948)). Возникает и новый компромисс — между вычислительной мощностью на инференсе и размером самой модели: иногда дешевле дать небольшой модели «подумать дольше», чем раздувать её параметры

Там же впервые наблюдался ставший известным A-ha moment - эффект, когда модель вдруг понимает, как правильно решать задачу, зачеркивает все написанное ранее и пишет правильное решение

<img src="img/cot5.png" width=300>

---

## Агентность

Рассуждение, замкнутое внутри модели, — это монолог. Агент начинается там, где модель замыкает цикл на внешний мир: формулирует мысль, совершает действие, получает наблюдение и корректирует следующий шаг. Именно этот цикл отличает агента от одиночного промпта.

### Паттерн ReAct

Парадигма **ReAct** ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)), предложенная в Google — отправная точка всего направления. Название склеено из *Reasoning* и *Acting*, и идея ровно в этом: чередовать рассуждение и действие. Языковая модель 
1) рассуждает («чтобы ответить, мне нужно узнать X»)
2) выбирает действие (например, поиск)
3) получает наблюдение (результат поиска)
4) go to 1: снова рассуждает уже с учётом нового факта

<img src="img/react.png" width=400>

В своих экспериментах авторы дали модели несколько примеров рассуждения в стилистике Reason-Act и попросили действовать в таком же ключе на новых заданиях. Так же они предоставили модели три функции для подгрузки информации из корпуса текстов Wiki: "[seach entity]", "[lookup string]", "[finish answer]".

Сравнивали свой "React prompting" со "Standard Prompting", "Chain-of-Thought prompting" и "Act-only prompting" (такое тоже есть - модель может только выполнять запросы, но ей запрещено рассуждать)

<img src="img/react1.png" width=750>

Тестировались на четырех задачах (все требуют multi-hop рассуждений):
- [HotSpotQA](https://arxiv.org/abs/1809.09600): поиск информации по базе знаний
- [Fever](https://arxiv.org/abs/1803.05355): поиск подтверждений или опровержения факта по базе знаний
- [ALFWorld](https://arxiv.org/abs/2010.03768): выполнить текстовое задание в "симуляторе" дворецкого
- [Webshop](https://arxiv.org/abs/2207.01206): купить товар на маркетплейсе по инструкции

Авторы честно показывают, что React не всегда превосходит CoT, а только на классе задач. ReAct сознательно отказывается от внутреннего знания в пользу внешнего поиска и его потолок упирается в качество этого поиска

<img src="img/react2.png" width=500>

Также авторы показывают важность дообучения ReAct модели через SFT. Модель довольно быстро обгоняет конкурентов, если дообучена не через few-shot промптинг

Направшиваются параллели с различными фреймворками когнитивной деятельности человека<br>
Например, PDCA из теории менеджмента: Plan -> Plan / Do / Check / Adjust
Или цикл действия разумных живых организмов: Perceive / Plan / Act / Reflex

---

### Использование инструментов (Tools)
Первые механические орудия (каменное рубило) появились около 3 млн лет назад. Они фактически заменили человеку хищные зубы, открыв доступ к высококалорийной пище. Главной предпосылкой стало развитие прямоходждения (Homo Erectus вышли из леса), развитие кистей, и появление абстрактоного мышления (орудие нужно еще изготовить). А высокая социализированность по сравнению с другими видами позволила поствить это умение "на конвейер". В результате вектор эволюции сместился: вместо замыкания на себе у вида Homo появилась возможность изменять окружающую среду

Похожая эволюция наблюдалась в развитии LLM агентов. Модели развились достаточно,чтобы строить многошаговые рассуждения, выполнять действия и менять входные данные.

Инструмент (Tool) - это любая внешняя по отношению к модели возможность изменить вход. Очевидные примеры инструментов:
- Web поиск<br>Задача обновления данных самых насущных, поэтому самый очевидный пример = веб-поиск.
- калькулятор

Как научить модель пользоваться инструментами?

Авторы **Toolformer** от Meta ([Schick et al., 2023](https://arxiv.org/abs/2302.04761)) решили сделать версию трансформера, реализующего такую механику, которая при этом была бы масштабируема. Идея: у нас есть некий пул инструментов, давайте научим модель самой определять, когда использовать инструмент, а когда нет, и главное сделать это без ручной разметки примеров, так как а) ручная разметка - это дорого б) разметка неочевидна => неизбежны ошибки

Два ключевых отличия от модели ReAct:
- здесь используется Self-supervised обучение (сам разметил - сам обучил)<br>все же нужно около 5 few-shot примеров показать, как использовать инструмента
- нет бесконечного цикла как в ReAct, тут предлагается разовое обогащение

Алгоритм обучения по принципу Self-supervised Rejection Sampling Fine-tuning
1. Construct Fine-tuning Dataset
    - берем текст из какого-либо корпуса текстов
    - в рандомные локации в тексте вставляются команды (не совсем рандомно, а pretrained модель во few-shot режиме выставляет где считает нужным)
    - замеряем кросс-энтропию справа от локации "без" / "с"<br>цель вставки - сократить неопределенность продолжения
    - если кросс-энтропия улучшилась больше чем на порог $\tau$ => оставляем, инструмент оказался полезен
2. Fine-tune Model to use Tools
    - делаем полноценное дообучение модели на расширенном датасете

<img src="img/toolformer.png" width=750>

То есть разметка все-таки нужна, но достаточно 5 few-shot примеров на каждый инструмент.

В качестве трансформера взяли открытую модель класса [GPT-J](https://en.wikipedia.org/wiki/GPT-J?utm_source=chatgpt.com)<br>использование инструментов абстрагируют через специальный токен `<API>`. В тестах использовали 5 инструментов: AtlasQA для фактических вопросов, Calculator, WikiSearch, Machine Translation, Calendar. Инференс жадный $\tau=0$ (за иключенем для токена <API>, там берется топ-10).

Замеряли свою модель на большом кол-ве бенчей: MLM LAMA, QA, MQA, Temporal Математика ASDIV, перевод, 

Кому интересно, есть и другие похожие подходы к обучению:
- __STaR__<br>Self-Taught Reasoner (Zelikman et al., 2022) — генерируем рассуждения, оставляем те, что дали верный ответ, дообучаемся, повторяем
- __RFT__<br>Метод Rejection Sampling Fine-Tuning, (Yuan et al., 2023) — то же, но сэмплируем много вариантов и фильтруем по проверяемому критерию
- __ReST__<br>(Gulcehre et al., 2023; Singh et al., 2023) — обобщённый цикл «Grow (насэмплировать) → Improve (отфильтровать по награде, дообучить)
- __RAFT__<br>(Reward-rAnked FineTuning, Dong et al., 2023)
- Исторический предок в RL-литературе: expert iteration (Anthony et al., 2017, та же идея испольщуется в AlphaZero)

---

**HuggingGPT** от Microsoft ([Shen et al., 2023](https://arxiv.org/abs/2303.17580)) показал, что можно использовать LLM как дирижера __LLM-as-a-Controller__: языковая модель выступает диспетчером, который разбивает задачу и распределяет подзадачи между множеством специализированных моделей и инструментов. Пример - когда в задании много модальностей

Рабочее название было Jarvis, но поскольку решили строить систему на базе крупнейшего открытого репозитория моделей HuggingFace, это и получило отражение в названии

Можно отметить и другие попытки построить оркестрацию моделей: Visual ChatGPT, ViperGPT, Chameleon, [Gorilla](https://arxiv.org/abs/2305.15334), [AutoGPT](https://arxiv.org/abs/2306.02224). Но HuggingGPT самый яркий пример

Использование моделями инструментов увеличивает инженерную сложность системы: как выбрать нужный инструмент из многих, как распарсить результат, как обработать ошибки и организовать повторные попытки при сбоях. 

*Инструменты формализуют действие (action) как вызов конкретной функции, а LLM из генератора текста превращается в управляющий контур, который маршрутизирует вызовы*

---

### Планирование

Для коротких задач реактивного цикла ReAct достаточно. Но на длинном горизонте агенту нужно заранее выстроить последовательность шагов, иначе он теряет нить.

Базовый паттерн - **Plan-and-Execute**. Один компонент (планировщик) строит план из шагов, другой (исполнитель) последовательно их выполняет, а при сбое запускается перепланирование. Уже знакомый Tree of Thoughts можно рассматривать как механизм планирования через поиск по дереву возможных траекторий

По аналогии с рассмотренным выше принципом Zero-shot Prompting, который прибавлял фразу "Let's think step--by-step", в работе __Plan-and-Execute__ [(Wang et al., 2023)](https://arxiv.org/abs/2305.04091)) авторы попытались еще улучшить поведение модели на  просто грамотной формулировкой задачи. Это назвали Plan-and-Solve Prompting. 

__Least-to-most prompting__ [(Zhou et al., 2022)](https://arxiv.org/abs/2205.10625)<br>
В Google предложили усовершенствовать CoT подход, чтобы лучше решались более сложные задачи. Они добавили шаг планирования: сначала требуем модель предоставить сгенерировать последоватльеный план решения с разбивкой на k подзадач, затем в отдельной генерации решаем каждую из подзадач. Выход одной является входом следующей. Последняя подзадача плана - выписать итоговый ответ<br><img src="img/least_to_most.png" width=500><br>Название откровенно идиотское, не понятно, что здесь Least, что Most

__Decomposed prompting__ [(Khot et al., 2022)](https://arxiv.org/abs/2210.02406)<br>
Модели говорят: сгенерируй структурированный план и отправь отдельные таски на выполнение. Добавляется рекурсия - если не обрабатывается за один проход, его разюивают на несколько подзадач

__Successive prompting__ [(Dua et al., 2022)](https://arxiv.org/abs/2212.04092)<br>
Итеративный способ рассуждения, при котором модель начинает с задавания себе уточняющих вопросов, пока не будет готова дать финальный ответ. Инструменты / внешние данные не предусмотрены

Похоже на итеративность ReAct, но ReAct - это про итеративность для обогащения внешними данными, а здесь про итеративность для более правильной логики получения ответа

__Compositionality gap__ / Self-Ask [(Press et al., 2022)](https://arxiv.org/abs/2210.03350)
Часто LLM модели страдают от того, что называют `назвал буквы, не смог собрать слово`. Предложили метод Self-ask - после каждой промехуточной генерации задавать Follow-up вопросы 

__LLM+P__ [(Liu et al., 2023)](https://arxiv.org/abs/2304.11477)
Метод LLM + Planning. Идея: модели говорят - сделай план решения, затем его конвертируют в специальный структурированный формат [PDDL](https://en.wikipedia.org/wiki/Planning_Domain_Definition_Language), и модель его выполняет. Задачами планирования действий ИИ занимались давно, язык из 90-х

Планирование вносит структуру в многошаговые задачи и разделяет стратегию (что делать) и исполнение (как делать)

---

### Память (Memory)

Стандартные LLM не хранят состояния между вызовами, взаимодейтсвие реализуется через контекст, который каждый раз читается полностью заново (а KV кэш?). Но контекстное окно конечно. При всех оптимизациях не бывает больше 1M токенов. Это фундаментальное ограничение: без памяти агент не может вести длинную задачу или помнить о прошлых взаимодействиях

«Lost in the Middle: How Language Models Use Long Contexts»
[(Liu et al., 2023)](arXiv:2307.03172)

Различают краткосрочную память (то, что помещается в текущий контекст) и долговременную (то, что хранится снаружи и подгружается по необходимости). 

Базовые приёмы 
- суммаризация истории (сжать прошлый диалог, чтобы он влез в окно),
- векторные хранилища (сохранять факты и доставать релевантные по семантическому поиску — основа RAG)
- аккуратное управление контекстным окном

Авторы модели **MemGPT** ([Packer et al., 2023](https://arxiv.org/abs/2310.08560)) попытались повторить логику работы ОС с оперативной памятью. Здесь LLM явно реализует двухуровневую модель памяти (оперативная vs постоянная) и сама решает, что держать в «оперативном» контексте, а что вытеснить во внешнее хранилище и подгрузить обратно при необходимости. Можно провести аналогию с [виртуальной памятью](https://en.wikipedia.org/wiki/Virtual_memory) и страничной подкачкой (page swap)

<img src="img/memgpt1.png" width=500>

Пример<br>

Работа закрепила новый термин "LLM as Operating System"

Наличие памяти позволяет работать с задачами длиннее, чем его контекстное окно

---

### Рефлексия и самообучение

Последний кирпичик - способность агента критиковать и улучшать самого себя, не дожидаясь внешнего обучения

**Reflexion** ([Shinn et al., 2023](https://arxiv.org/abs/2303.11366))

Идея: давайте использовать отдельную модель в качестве оценщика Verifier, а корректировать не путем обновления весов (как в RL), а просто добавлять в промпт текстовую поправку

<img src="img/reflection2.png" width=300>

**Voyager** ([Wang et al., 2023](https://arxiv.org/abs/2305.16291))<Br>довёл идею до пожизненного обучения на примере агента в Minecraft. Три механизма работают вместе: автоматический учебный план (агент сам ставит себе посильно усложняющиеся цели), библиотека навыков (удачные решения сохраняются как переиспользуемый код и комбинируются в более сложные) и обратная связь от среды-песочницы. Так агент постепенно накапливает компетенцию, а не решает каждую задачу с нуля

Агент становится самоулучшающимся в пределах сессии или «жизни», а библиотека навыков превращает разовые решения в композиционную, переиспользуемую компетенцию

---

<img src="img/agent_papers.jpg" width=500>

---



# Агентские системы
Зачем использовать один агент, если можно запрячь чразу много. Один агент перегружается задачами, путает роли, теряет контроль над длинным процессом. Тут правда сразу возникает ворох вопросов, связанных с тем, как именно разделять работу и как ими управлять. Всё это исследуется в рамках направления "мультиагентные системы"

Более поздние работы однако показали, что польза не такая однозначная. Например, здесь [(Tran et al, 2026)](https://arxiv.org/pdf/2604.02460) исследователи из Стенфорда показывают, что при фиксированном бюджете одноагентная конфигурация может быть ни чуть не хуже. Или здесь [(Xu et al, 2026)](https://arxiv.org/pdf/2601.12307) авторы исследуют важность узкой специализации и опровергают её, показав что ту же систему можно реализовать одной моделью, назначая ей роли промптами

Тем не менее в многих ситуациях разделение труда необходимо и на момент появления данного направления (около 2023) его оценивали, как крайне перспективное

Множество агентов в одной системе системе:
- улучшает "широту" мышления (Divergent Thinking)
- повышает точность ответов (improves factuality & reasoning)
- лучше валидирует ответ (Validation)

Автономность агентов влечет риски:
- каскдные ошибки
- prompt injection
- зациливаемость
и т.д.

Как определяется приоритеность команд? Например, приказ смежного агента противоречит системному промпту<br>
сисемный промпт

Как определяется порядок выполнения в мультагентных системах:
- жестко по графу<br>пример - игра в мафию
- агент сам определяет (push режим)<br>игра в мяч
- GroupChatManager - коорднатор<br>модератор на панельной сессии
- по условию / триггеру<br>

Агенты могут выполняться параллельно, но их координация дискретна - нужно должаться полного ответа перед дальнейшим действием

Часто цитируют две работы, описавшие стандарт для мультиагентных систем

---

### AutoGen

В 2023 году Microsoft предложил свой фреймворк **AutoGen** ([Wu et al., 2023](https://arxiv.org/abs/2308.08155)), описывающий механизмы решения задач агентами в формате разговора друг с другом

Работа популяризировала термины:<br>
Conversable Agent (диалоговый агент) - модель, готовая коммуницировать с внешним миром через язык<br>
Conversable Programming (диалоговое программирование) - способ координации работы между агентами при решении ими распределенной задачи, когда задачи ставятся текстом

Ценность работы в том, что она одной из первых показала, как можно эффективно организовывать работу множества агентов, просто дав им возможность разговаоривать на естественном языке

В работе вводят несколько классов агентов 
- AssistantAgent - "мозги" для генерации рассуждения
- UserProxyAgent - "руки" для автономного выполнения действий. Моделирует "что бы сделал пользователь"
- GroupChatManager - (опционально) координатор, выбирает, кто будет следующий действовать

<img src="img/autogen1.png" width=600>

Они протестировали свой подход на 6 задачах: математика, Information Retrieval, AFL, Multi-agent Coding, Multi agent chat, Conversational Chess

Фреймворк не только теоретический, он вполне используется на практике, выходят новые версии<br>https://github.com/microsoft/autogen

В версии 0.4, например, фреймворк переписали для воплощения модели вычислений Actor-model

#### Actor-model
[Actor Model](https://en.wikipedia.org/wiki/Actor_model) - это математическая модель асинхронных вычислений. Актор - это объект, который умеет а) принимать сообщения б) отправлять сообщения в) порождать дургих акторов г) выполнять какое-то действие

Модель разрабатывали еще с 1973 года. На основе нее были реализованы многие инженерные фреймворки для разработки Message-driven applications, например, фреймворк [Akka](https://en.wikipedia.org/wiki/Akka_(toolkit)) для Java

Похоже на концепцию микросервисов: вычисление тоже бъется на отдельные независимые куски, запускаемые асинхронно, но разница в уровне детализации. Akka работает на уровне отдельных объектов

### MetaGPT

В работе **MetaGPT** ([Hong et al., 2023](https://arxiv.org/abs/2308.00352)) принцип тот же, но тут разделение труда более стандартизированный. Там пошли от метафоры организации: каждый агент выполняет свою роль, как это бывает в софтверной компании: продакт-менеджер, архитектор, инженер

<img src="img/metagpt1.png" width=500>

### Оркестрация и архитектура

Тема была бы неполной без рисков, которые порождает множественность: агенты могут зациклиться в бесконечном диалоге, стоимость растёт пропорционально числу вызовов, а ошибка одного агента способна распространиться по всей системе

По мере усложнения систем линейных цепочек становится мало — нужен переход к управляемым графам состояний. Графовая оркестрация (типичный представитель — [LangGraph](https://github.com/langchain-ai/langgraph)) описывает агента как граф: узлы — это шаги, рёбра — переходы, есть общее состояние и допустимы циклы. Это даёт явный контроль над тем, что и в каком порядке происходит.

Хороший словарь паттернов задаёт статья Anthropic [«Building Effective Agents»](https://www.anthropic.com/engineering/building-effective-agents). Она проводит важное различие между workflow (заранее заданные маршруты, по которым модель ведут жёстко) и собственно агентами (модель сама управляет своим процессом). И описывает базовые композиционные паттерны: chaining (последовательная цепочка), routing (маршрутизация запроса в нужную ветку), параллелизация, оркестратор-исполнители, оценщик-оптимизатор

Сквозь всю тему проходит центральный компромисс проектирования: автономность против контроля. Чем больше свободы у модели, тем шире её возможности — и тем труднее предсказать и отладить поведение. Практический совет, который из этого следует: давать ровно столько автономности, сколько нужно для решения задачи, и не больше

### Стандарты подключения (MCP)
Когда инструментов и источников данных становится много, остро встаёт вопрос интеграции: каждый раз писать «переходник» между моделью и очередным сервисом — дорого и не масштабируется. Ответ — стандартизация

[Model Context Protocol (MCP)](https://modelcontextprotocol.io) — это открытый протокол стандартизированного подключения LLM к инструментам и данным; его можно визуализировать как «USB порт для ИИ». 

Появляются серверы инструментов и источники данных, которые реализуют протокол один раз, после чего любой совместимый агент может ими пользоваться. Тема здесь не столько техническая, сколько системная: почему стандарт важен для масштаба — он развязывает интеграцию и приложение, и за счёт сетевого эффекта порождает целую экосистему переиспользуемых компонентов

Тема новая и протоколы пока не устаявшиеся, поэтому попытались попытались навязать войну стандартов: A2A от Google, MCP. На текущий момент пока стнадартом ялвяется MCP

### Фреймворки
Инструмент нужно выбирать под задачу, а не привязываться к одному. Полезно держать в голове грубую карту:
- LangGraph - про управление и контроль через графы состояний; силён там, где важны циклы и явная логика переходов
- LlamaIndex ([github](https://github.com/run-llama/llama_index)) — про данные и RAG; силён в индексации и извлечении знаний.
- AutoGen - про мульти-агентное взаимодействие в формате разговора
- CrewAI ([github](https://github.com/crewAIInc/crewAI)) — про ролевые «команды» агентов с понятным разделением ролей

Иногда фреймворк не нужен вовсе. Та же Anthropic советует начинать с простого - прямых вызовов модели - и добавлять сложность только тогда, когда она реально окупается

---



## Production

Продакшен предъявляет четыре требования, которые в исследовательских работах часто остаются за скобками: уметь измерять успех, видеть происходящее внутри, защищать агента и делать его экономичным

### Оценка агентов

Без измерения невозможна осмысленная итерация. Сложность в том, что агент недетерминирован, проходит много шагов и часто заслуживает «частичного зачёта», — простой accuracy здесь не работает.

В арсенале 
— метрики успешности задач (довёл ли агент дело до конца) 
- LLM-as-a-judge (использовать другую модель как оценщика качества)
- специализированные бенчмарки:
    - [WebArena](https://arxiv.org/abs/2307.13854) (2023) для задач в вебе,
    - [GAIA](https://arxiv.org/abs/2311.12983) для ассистентов общего назначения,
    - [τ-bench (TauBench)](https://arxiv.org/abs/2406.12045) для взаимодействия с инструментами и пользователем в реалистичных доменах.

Отдельно стоит оценка RAG-компонентов — например, через [RAGAS](https://aclanthology.org/2024.eacl-demo.16), который измеряет качество извлечения и достоверность ответа

### Наблюдаемость и отладка
Нужно видеть, что происходит внутри многошагового процесса. Базовые инструменты — трейсинг всех вызовов (модели и инструментов), логирование шагов рассуждения и средства отладки недетерминированных сценариев, где один и тот же вход может приводить к разным траекториям

### Безопасность

Агент, способный выполнять реальные действия, — это реальная поверхность атаки, и здесь ставки выше, чем у обычного чат-бота. Центральная угроза — **prompt injection** и **jailbreak**: вредоносные инструкции, спрятанные во входных данных или на веб-странице, которые перехватывают поведение агента. Защитные принципы пришли из классической безопасности: изоляция инструментов (песочница), принцип минимальных прав (агент получает ровно те доступы, что нужны) и ограничение области действия агента.

### Оптимизация

Чтобы агента можно было реально развернуть, он должен быть достаточно дешёвым и быстрым. Главные рычаги — контроль стоимости и латентности, кэширование (в том числе кэш промптов, чтобы не пересчитывать повторяющийся контекст), выбор модели под подзадачу (мелкая быстрая модель на простые шаги, крупная — только там, где нужна) и маршрутизация запросов между моделями разной мощности

### Передний край: агентский RL

Если в части 1 мы учили модель рассуждать с помощью RL, то теперь та же логика применяется к агентам целиком: вместо ручной настройки промптов и оркестрации — обучение агента через RL на взаимодействии со средой, end-to-end. Сюда же относятся **самоулучшающиеся агенты** и множество открытых проблем (стабильность обучения, награды на длинном горизонте, перенос между задачами). 

Обзорная точка входа в тему — [«The Landscape of Agentic Reinforcement Learning for LLMs: A Survey»](https://arxiv.org/abs/2509.02547) (2025)

---



## Резюме

Итого мы научились 
- заставлять модель думать (CoT) →
- думать надёжнее через поиск по вариантам (Self-Consistency, ToT) →
- обучать думать (reasoning-модели, RLVR) →
- заземлили мышление на действие (ReAct) →
- формализовали действия как вызовы инструментов (Toolformer, function calling) →
- внесли структуру через планирование, дали агенту **непрерывность** через память (MemGPT) и **способность улучшаться** через рефлексию (Reflexion, Voyager) →
- распределили работу между агентами (AutoGen, MetaGPT) →
- взяли поток под контроль через графовую оркестрацию (LangGraph, паттерны Anthropic) →
- стандартизировали подключение (MCP) →
- наконец, закалили систему для продакшена через оценку, наблюдаемость, безопасность и оптимизацию →
- агентов начинают обучать целиком через RL

---

## Хронологическая шпаргалка

Компактная карта идей — удобно для сжатия в слайды и для собеседований.

| Идея / работа | Год | Ключевая мысль | Что открыло дальше |
|---|---|---|---|
| Chain-of-Thought | 2022 | Проговаривать шаги перед ответом | Рассуждение как управляемый объект |
| ReAct | 2022 | Чередовать мысль и действие | Цикл «мысль → действие → наблюдение», предтеча function calling |
| Self-Consistency | 2022 | Много цепочек + голосование | Качество за счёт вычислений |
| Toolformer / HuggingGPT | 2023 | Модель сама вызывает инструменты; LLM-диспетчер | Формализация tool use |
| Tree of Thoughts | 2023 | Рассуждение как поиск по дереву | Планирование через поиск |
| Reflexion | 2023 | Вербальная рефлексия над ошибками | Самообучение без обновления весов |
| Voyager | 2023 | Автокурс + библиотека навыков | Композиционная, переиспользуемая компетенция |
| MemGPT | 2023 | Память как у ОС (подкачка) | Непрерывность за пределами окна контекста |
| AutoGen / MetaGPT | 2023 | Агенты-разговор; роли как в компании | Мульти-агентные системы |
| MCP | 2024 | Стандарт подключения инструментов и данных | Экосистема и масштаб |
| Building Effective Agents | 2024 | Workflow vs. агент; минимум автономности | Дисциплина оркестрации |
| Reasoning-модели / RLVR | 2024–2025 | Обучать рассуждать; масштабирование на инференсе | Сильные агенты как обучаемая способность |
| Agentic RL (survey) | 2025 | Обучать агента целиком через RL | Передний край: самоулучшающиеся агенты |

---

Поле меняется быстро, поэтому статичный список устаревает. Полезны постоянно обновляемые источники: ежегодные обзоры статей Себастьяна Рашки, reading list от Latent.Space и GitHub-репозитории `zjunlp/LLMAgentPapers` и `AGI-Edgerunners/LLM-Agents-Papers`.